# NeMo Data Designer – Synthetic Data Generation Example

This notebook demonstrates how to use **NeMo Data Designer** (NDD) stage to generate synthetic medical-notes data from a small seed dataset.

The pipeline:
1. Downloads a public symptom-to-diagnosis CSV and converts it to JSONL shards.
2. Define the configuration for the NDD stage.
3. Build and run a Curator pipeline, composing of three stages: `JsonlReader`, `DataDesignerStage`, `JsonlWriter`.

In [ ]:
import sys
!{sys.executable} -m pip install -e ../../..
uv sync --extra sdg_cpu

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///lustre/fsw/coreai_dlalgo_genai/huvu/codes/nemo_data_designer/NeMo-Curator_ndd_ipynb_tutorials
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nemo-curator (pyproject.toml) ... done
  Created wheel for nemo-curator: filename=nemo_curator-1.1.0rc0.dev0-0.editable-py3-none-any.whl size=11057 sha256=9fb489e82736dffa8a0eea861a1506abba3fa7bcd22bfb2e241aba475c6873fd
  Stored in directory: /tmp/pip-ephem-wheel-cache-mh_0t9la/wheels/73/72/42/e81d091c3e7d640f229fce294910682bba9015c051dc950c10
Successfully built nemo-curator
  Attempting uninstall: nemo-curator
    Found existing installation: nemo-curator 1.1.0rc0.dev0
    Uninstalling nemo-curator-1.1.0rc0.dev0:
      Successfully uninstalled nemo-curator-1.1.0rc0.dev0


### Imports libraries

In [ ]:
import time
from pathlib import Path

import data_designer.config as dd
import pandas as pd

from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.synthetic.nemo_data_designer.data_designer import DataDesignerStage
from nemo_curator.stages.text.io.reader.jsonl import JsonlReader
from nemo_curator.stages.text.io.writer.jsonl import JsonlWriter
from nemo_curator.utils.file_utils import get_all_file_paths_under

/home/huvu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-06 08:56:29,440	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


### Download and Prepare Seed Data

The seed dataset is a public symptom-to-diagnosis CSV hosted on GitHub.  
We download it, split it into small JSONL shards (10 rows each), and save them locally so the pipeline can read them in parallel.

In [4]:
SEED_CSV_URL = (
    "https://raw.githubusercontent.com/NVIDIA/GenerativeAIExamples/refs/heads/main"
    "/nemo/NeMo-Data-Designer/data/gretelai_symptom_to_diagnosis.csv"
)


def download_and_convert_seed_data(
    output_dir: str | Path | None = None,
    records_per_file: int = 10,
) -> str:
    """Download seed CSV from URL, convert to JSONL (chunked), return output dir path."""
    if output_dir is None:
        output_dir = Path("processed_seed_data")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(SEED_CSV_URL, sep=",", encoding="utf-8")
    for i, start in enumerate(range(0, len(df), records_per_file)):
        chunk = df.iloc[start : start + records_per_file]
        chunk.to_json(
            output_dir / f"{i:06d}.jsonl",
            orient="records",
            lines=True,
            force_ascii=False,
            date_format="iso",
        )
    return str(output_dir)


print("Preparing seed data (download + CSV→JSONL)...")
seed_dir = download_and_convert_seed_data()
print(f"Seed data ready: {seed_dir}")

Preparing seed data (download + CSV→JSONL)...
Seed data ready: processed_seed_data


### Configuration

Set the output directory for generated data here.  
You can also point `DATA_DESIGNER_CONFIG_FILE` at a YAML config file; leave it as `None` to use the programmatic config defined in this notebook.

In [5]:
OUTPUT_PATH = "./synthetic_output"
DATA_DESIGNER_CONFIG_FILE = None  # set to a file path string to load a YAML config

### Define the Data Designer Config

The DataDesignerStage is intialized from a NDD's configuration builder (https://nvidia-nemo.github.io/DataDesigner/latest/code_reference/config_builder/). In this step, we manually create a configuration for the NDD module. This includes setting the LLM model provider, and the steps taken inside NDD module. Users can learn more about NDD's API from NDD official documentation: https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/.

In this tutorial, we replicate the official NDD's tutorials on synthetic data genertion with seed data (https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/), but with Nemo-Curator.


In [6]:
def build_config_manually() -> dd.DataDesignerConfigBuilder:
    """Build the Data Designer config programmatically for medical-notes generation."""
    model_provider = "nvidia"
    model_id = "meta/llama-3.3-70b-instruct"
    model_alias = "llama-3.3-70b"

    model_configs = [
        dd.ModelConfig(
            alias=model_alias,
            model=model_id,
            provider=model_provider,
            inference_parameters=dd.ChatCompletionInferenceParams(
                temperature=1.0,
                top_p=1.0,
                max_tokens=2048,
            ),
        )
    ]

    config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

    # --- Sampler columns (faker / random) ---
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="patient_sampler",
            sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
            params=dd.PersonFromFakerSamplerParams(),
        )
    )
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="doctor_sampler",
            sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
            params=dd.PersonFromFakerSamplerParams(),
        )
    )
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="patient_id",
            sampler_type=dd.SamplerType.UUID,
            params=dd.UUIDSamplerParams(prefix="PT-", short_form=True, uppercase=True),
        )
    )

    # --- Expression columns (Jinja2 references) ---
    config_builder.add_column(dd.ExpressionColumnConfig(name="first_name", expr="{{ patient_sampler.first_name}}"))
    config_builder.add_column(dd.ExpressionColumnConfig(name="last_name",  expr="{{ patient_sampler.last_name }}"))
    config_builder.add_column(dd.ExpressionColumnConfig(name="dob",        expr="{{ patient_sampler.birth_date }}"))

    # --- Date / timedelta samplers ---
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="symptom_onset_date",
            sampler_type=dd.SamplerType.DATETIME,
            params=dd.DatetimeSamplerParams(start="2024-01-01", end="2024-12-31"),
        )
    )
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="date_of_visit",
            sampler_type=dd.SamplerType.TIMEDELTA,
            params=dd.TimeDeltaSamplerParams(
                dt_min=1,
                dt_max=30,
                reference_column_name="symptom_onset_date",
            ),
        )
    )
    config_builder.add_column(
        dd.ExpressionColumnConfig(name="physician", expr="Dr. {{ doctor_sampler.last_name }}")
    )

    # --- LLM-generated column ---
    config_builder.add_column(
        dd.LLMTextColumnConfig(
            name="physician_notes",
            prompt="""\
You are a primary-care physician who just had an appointment with {{ first_name }} {{ last_name }},
who has been struggling with symptoms from {{ diagnosis }} since {{ symptom_onset_date }}.
The date of today's visit is {{ date_of_visit }}.

{{ patient_summary }}

Write careful notes about your visit with {{ first_name }},
as Dr. {{ doctor_sampler.first_name }} {{ doctor_sampler.last_name }}.

Format the notes as a busy doctor might.
Respond with only the notes, no other text.
""",
            model_alias=model_alias,
        )
    )

    return config_builder

### Build the NeMo Curator Pipeline

After defining the NDD configuration which will be used to initialize `DataDesignerStage` stage, we build the pipeline chains three stages:

| # | Stage | Role |
|---|-------|------|
| 1 | `JsonlReader` | Read seed JSONL shards from disk |
| 2 | `DataDesignerStage` | Enrich each record using the NDD config |
| 3 | `JsonlWriter` | Write enriched records back to JSONL |

In [7]:
pipeline = Pipeline(
    name="ndd_data_generation",
    description="Generate synthetic text data using Nemo Data Designer",
)

pipeline.add_stage(
    JsonlReader(
        file_paths=seed_dir + "/*.jsonl",
        fields=["diagnosis", "patient_summary"],
    )
)

if DATA_DESIGNER_CONFIG_FILE is not None:
    config_builder = dd.DataDesignerConfigBuilder.from_config(DATA_DESIGNER_CONFIG_FILE)
else:
    config_builder = build_config_manually()

pipeline.add_stage(DataDesignerStage(config_builder=config_builder))

pipeline.add_stage(JsonlWriter(path=OUTPUT_PATH))

print(pipeline.describe())

2026-03-06 08:56:56.836 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'jsonl_reader' to pipeline 'ndd_data_generation'
2026-03-06 08:57:00.097 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'DataDesignerStage' to pipeline 'ndd_data_generation'
2026-03-06 08:57:00.102 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'jsonl_writer' to pipeline 'ndd_data_generation'


Pipeline: ndd_data_generation
Description: Generate synthetic text data using Nemo Data Designer
Stages: 3

Stage 1: jsonl_reader
  Resources: 1.0 CPUs
  Batch size: 1
  Outputs:
    Output attributes: data
    Output columns: diagnosis, patient_summary
Stage 2: DataDesignerStage
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required attributes: data
  Outputs:
    Output attributes: data
Stage 3: jsonl_writer
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required attributes: data
  Outputs:
    Output attributes: data



### Run the Pipeline

In [8]:
import os
os.environ["NVIDIA_API_KEY"] = "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"  # your key from build.nvidia.com

In [ ]:
client = RayClient()  # change as needed
client.start()

In [ ]:
print("Starting synthetic data generation pipeline...")
start_time = time.time()
pipeline.run()
end_time = time.time()

elapsed_time = end_time - start_time
print(f"\nPipeline completed!")
print(f"Total execution time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)")

2026-03-06 08:57:08.666 | INFO     | nemo_curator.pipeline.pipeline:build:70 - Planning pipeline: ndd_data_generation
2026-03-06 08:57:08.667 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:106 - Decomposing composite stage: jsonl_reader
2026-03-06 08:57:08.668 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:120 - Expanded 'jsonl_reader' into 2 execution stages


Starting synthetic data generation pipeline...


2026-03-06 08:57:09.111 | INFO     | nemo_curator.backends.xenna.executor:execute:135 - Execution mode: STREAMING
2026-03-06 08:57:27,411	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


### Inspect the Output

In [ ]:
output_files = get_all_file_paths_under(OUTPUT_PATH, recurse_subdirectories=True, keep_extensions=".jsonl")

print(f"Generated data saved to: {OUTPUT_PATH}")
for file_path in output_files:
    print(f"  - {file_path}")

all_data_frames = [pd.read_json(f, lines=True) for f in output_files]

In [ ]:
print("=" * 50)
print("Sample of generated documents:")
print("=" * 50)

for i, df in enumerate(all_data_frames):
    print(f"\nFile {i + 1}: {output_files[i]}")
    print(f"Number of documents: {len(df)}")
    print("\nGenerated text (showing first 5):")
    for j, row in enumerate(df.head(5).to_dict(orient="records")):
        print(f"Document {j + 1}:")
        for key, value in row.items():
            print(f"[{key}]:")
            print(f"{value}")
        print("-" * 40)

### Preview as a DataFrame

In [ ]:
if all_data_frames:
    combined_df = pd.concat(all_data_frames, ignore_index=True)
    print(f"Total records generated: {len(combined_df)}")
    combined_df.head()